In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [15]:
# Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  ✗ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents


# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 4 PDF files to process

Processing: god_of_war.pdf
  ✓ Loaded 2 pages

Processing: Notes.pdf
  ✓ Loaded 27 pages

Processing: RESUMEE_bharat.pdf
  ✓ Loaded 1 pages

Processing: sekiro.pdf
  ✓ Loaded 3 pages

Total documents loaded: 33


In [16]:
all_pdf_documents

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-17T08:52:43+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-17T08:52:43+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\god_of_war.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'god_of_war.pdf', 'file_type': 'pdf'}, page_content="GOD OF WAR\n The Complete Guide — Legacy, Lore & Legacy\nOverview\nGod of War is one of the most acclaimed action-adventure video game franchises in history, developed by\nSanta Monica Studio and published by Sony Interactive Entertainment. Beginning in 2005, the series\nfollows Kratos — a Spartan warrior turned Greek God of War — through mythologies spanning Greece\nand Norse worlds. Renowned for its epic storytelling, visceral combat, and stunning visuals, God of War\nhas defined a generation of gaming.\nGame Timeline\nYear\nTitle\nSe

In [22]:
# Text splitting get into chunks

def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [23]:
chunks = split_documents(all_pdf_documents)
chunks

Split 33 documents into 51 chunks

Example chunk:
Content: GOD OF WAR
 The Complete Guide — Legacy, Lore & Legacy
Overview
God of War is one of the most acclaimed action-adventure video game franchises in history, developed by
Santa Monica Studio and publishe...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-17T08:52:43+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-17T08:52:43+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\god_of_war.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'god_of_war.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-17T08:52:43+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-17T08:52:43+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\god_of_war.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'god_of_war.pdf', 'file_type': 'pdf'}, page_content="GOD OF WAR\n The Complete Guide — Legacy, Lore & Legacy\nOverview\nGod of War is one of the most acclaimed action-adventure video game franchises in history, developed by\nSanta Monica Studio and published by Sony Interactive Entertainment. Beginning in 2005, the series\nfollows Kratos — a Spartan warrior turned Greek God of War — through mythologies spanning Greece\nand Norse worlds. Renowned for its epic storytelling, visceral combat, and stunning visuals, God of War\nhas defined a generation of gaming.\nGame Timeline\nYear\nTitle\nSe

### Embedding and VectorStoreDB

In [24]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [28]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        
        Args:
            model_name: HuggingFace model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(
                f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts
        
        Args:
            texts: List of text strings to embed
            
        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings


# initialize the embedding manager

embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


c:\Users\Tarunvarma\OneDrive\Desktop\demo-RAG\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Tarunvarma\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2371.33it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\Tarunvarma\AppData\Local\Temp\ipykernel_2868\3130810938.py:21: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")


In [29]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        
        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(
                path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(
                f"Vector store initialized. Collection: {self.collection_name}")
            print(
                f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(
                f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise


vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [30]:
chunks

[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-17T08:52:43+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-05-17T08:52:43+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': '..\\data\\pdf\\god_of_war.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'god_of_war.pdf', 'file_type': 'pdf'}, page_content="GOD OF WAR\n The Complete Guide — Legacy, Lore & Legacy\nOverview\nGod of War is one of the most acclaimed action-adventure video game franchises in history, developed by\nSanta Monica Studio and published by Sony Interactive Entertainment. Beginning in 2005, the series\nfollows Kratos — a Spartan warrior turned Greek God of War — through mythologies spanning Greece\nand Norse worlds. Renowned for its epic storytelling, visceral combat, and stunning visuals, God of War\nhas defined a generation of gaming.\nGame Timeline\nYear\nTitle\nSe

In [31]:
texts=[doc.page_content for doc in chunks]
texts

["GOD OF WAR\n The Complete Guide — Legacy, Lore & Legacy\nOverview\nGod of War is one of the most acclaimed action-adventure video game franchises in history, developed by\nSanta Monica Studio and published by Sony Interactive Entertainment. Beginning in 2005, the series\nfollows Kratos — a Spartan warrior turned Greek God of War — through mythologies spanning Greece\nand Norse worlds. Renowned for its epic storytelling, visceral combat, and stunning visuals, God of War\nhas defined a generation of gaming.\nGame Timeline\nYear\nTitle\nSetting\nPlatform\n2005\nGod of War\nAncient Greece\nPS2\n2007\nGod of War II\nAncient Greece\nPS2\n2010\nGod of War III\nMount Olympus\nPS3\n2013\nGod of War: Ascension\nGreece (Prequel)\nPS3\n2018\nGod of War (Reboot)\nNorse Mythology\nPS4\n2022\nGod of War: Ragnarok\nNorse — Nine Realms\nPS4/PS5\nKratos — The Ghost of Sparta\nKratos is one of gaming's most iconic protagonists. Born a Spartan warrior, he was tricked by Ares into",
 "PS4\n2022\nGod of W

In [32]:
# Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

# Generate the Embeddings

embeddings = embedding_manager.generate_embeddings(texts)

# store int he vector dtaabase
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 51 texts...


Batches: 100%|██████████| 2/2 [00:07<00:00,  3.68s/it]


Generated embeddings with shape: (51, 384)
Adding 51 documents to vector store...
Successfully added 51 documents to vector store
Total documents in collection: 51


### retriever

In [33]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager
    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore,embedding_manager)

In [34]:
rag_retriever

In [38]:
rag_retriever.retrieve("Awards & Recognition of god of war")

Retrieving documents for query: 'Awards & Recognition of god of war'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  1.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_1cb6b707_4',
  'content': 'beloved internet meme.\n\x7f The 2018 game\'s single-shot camera was inspired by films like Birdman and 1917.\n\x7f Atreus is revealed to be the Norse trickster god Loki — his mother was a giant named Faye (Laufey).\n\x7f God of War: Ragnarok features over 75 playable characters and enemies drawn from Norse legends.\n "The cycle ends here. We must be better than this." — Kratos, God of War (2018)\n Compiled for fans of the God of War franchise. All rights to respective owners.',
  'metadata': {'content_length': 466,
   'page': 1,
   'creator': '(unspecified)',
   'creationdate': '2026-05-17T08:52:43+00:00',
   'doc_index': 4,
   'moddate': '2026-05-17T08:52:43+00:00',
   'author': '(anonymous)',
   'keywords': '',
   'subject': '(unspecified)',
   'source_file': 'god_of_war.pdf',
   'title': '(anonymous)',
   'page_label': '2',
   'file_type': 'pdf',
   'source': '..\\data\\pdf\\god_of_war.pdf',
   'trapped': '/False',
   'total_pages': 2,
   'pr

In [39]:
rag_retriever.retrieve("whats story of sekiro shadows die twice?")

Retrieving documents for query: 'whats story of sekiro shadows die twice?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 33.56it/s]

Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)


[{'id': 'doc_c7f3a0b2_44',
  'content': "SEKIRO\n Shadows Die Twice — The Complete Guide\nOverview\nSekiro: Shadows Die Twice is an action-adventure game developed by FromSoftware and published by\nActivision, released on March 22, 2019. Set in a reimagined late Sengoku-era Japan, it follows a shinobi\nnamed Wolf (Ookami) on a mission to rescue his kidnapped lord and take revenge on his enemies. Unlike\nFromSoftware's Souls series, Sekiro emphasizes precise sword combat, stealth, and vertical traversal\nusing a grappling hook prosthetic arm. It won the prestigious Game of the Year 2019 at The Game\nAwards.\nQuick Facts\nDetail\nInfo\nDeveloper\nFromSoftware\nPublisher\nActivision\nRelease Date\nMarch 22, 2019\nPlatforms\nPS4, Xbox One, PC, Stadia\nGenre\nAction-Adventure / Stealth\nSetting\nLate Sengoku-era Japan (1500s)\nProtagonist\nWolf (Ookami) — The One-Armed Wolf\nGOTY\nThe Game Awards 2019\nStory & Setting\nThe game takes place in a fictionalised version of Sengoku-period Japan,

### Integration Vectordb Context pipeline With LLM output

In [43]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="qwen/qwen3-32b",temperature=0.1,max_tokens=1024)


def rag_simple(query, retriever, llm, top_k=3):
    # retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content']
                          for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    # generate the answwer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content



In [51]:
answer = rag_simple("is sekiro the GOAT?", rag_retriever, llm)
print(answer)

Retrieving documents for query: 'is sekiro the GOAT?'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 22.04it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


<think>
Okay, the user is asking if Sekiro is the GOAT, which stands for Greatest Of All Time. Let me start by recalling the context provided.

First, the context mentions that Sekiro's name means "One-Armed Wolf" in Japanese. The game was directed by Hidetaka Miyazaki, known for other FromSoftware titles like Dark Souls and Bloodborne. The game is inspired by samurai films and manga, which gives it a unique aesthetic and narrative style. It's noted that Sekiro has a fixed protagonist, Wolf, which is a first for FromSoftware's modern era, meaning players don't get to create their own character. Also, it's a purely solo experience with no multiplayer elements. The Guardian Ape's second phase was a big moment in 2019, and there's a speedrun under 20 minutes.

Now, to determine if it's the GOAT, I need to consider its strengths. The game's unique setting, inspired by samurai media, sets it apart. The fixed protagonist and solo focus might appeal to fans of the genre. The Guardian Ape boss

### Enhanced RAG Pipeline Features

In [53]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output


# Example usage:
result = rag_advanced("what is god of war", rag_retriever,
                      llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'what is god of war'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:01<00:00,  1.20s/it]


Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Answer: <think>
Okay, let's tackle this question. The user is asking "what is god of war" and wants a concise answer based on the provided context.

First, I need to recall the key points from the context. The context mentions that God of War is an action-adventure franchise by Santa Monica Studio, published by Sony. It started in 2005 and follows Kratos, a Spartan turned Greek God of War. The series spans Greek and Norse mythologies. The 2018 game was a reboot set in Norse, and Ragnarok in 2022. It's known for storytelling, combat, and visuals. Also, Kratos's son Atreus is Loki, and there are mentions of awards and some fun facts like the single-shot camera inspired by films.

The user wants a concise answer. So I should start by stating it's a video game franchise, developed by Santa Monica Studio, published by Sony. Mention Kratos as the protagonist, the transition from Greek to Norse mythology, and hi

In [57]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(
            question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke(
                [prompt.format(context=context, question=question)])
            answer = response.content

        # Add citations to answer
        citations = [
            f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + \
            "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }


# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("who is the final villain in sekiro?",
                       top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'who is the final villain in sekiro?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
 The name 'Sekiro' means 'One-Armed Wolf' in Japanese (Seki = one-armed, ro = wolf).
 Director Hidetaka Miyazaki said Sekiro was inspired by old Japanese samurai films and manga.
 The game has no character creation — Wolf is a fixed protagonist, a 

first for FromSoftware's modern
era.
 Sekiro has no multiplayer — no co-op, no invasions. It is a purely solo experience.
 The Guardian Ape's second phase (headless) became one of the most shocked-about gaming
moments of 2019.
 A speedrunner completed Sekiro in under 20 minutes using precise knowledge of boss patterns.
 "I am Wolf. I do not fear death." — Sekiro: Shadows Die Twice
 Compiled as a fan guide. All rights belong to FromSoftware and Activision.

Question: who is the final villain in sekiro?

Answer:

Final Answer: <think>
Okay, let's see. The user is asking who the final villain in Sekiro is. I need to use the provided context to answer this. Let me check the context again.

The context mentions the Guardian Ape's second phase being a shocking moment in 2019. The Guardian Ape is a boss in the game. But is that the final villain? Wait, the final boss in Sekiro is usually considered Genichiro Ashina. But the context doesn't mention Genichiro. The context talks about the Gua